In [1]:
import sys
import os
from dotenv import load_dotenv, find_dotenv
import geopandas as gpd
import matplotlib.pyplot as plt
import fiona
from shapely.geometry import box
import pandas as pd

load_dotenv(find_dotenv())

DATA_PATH = os.getenv("DATA_PATH")
DEWEY_PATH = os.getenv("DEWEY_PATH")

In [2]:
# Create a list of all .parquet files in DEWEY_PATH
parquet_files = [f for f in os.listdir(DEWEY_PATH) if f.endswith('.parquet')]

In [3]:
summ_dfs = []
for i, f in enumerate(parquet_files):
    print(f"\rProcessing {f}: ({i+1}/{len(parquet_files)}) ... ", end="", flush=True)
    df = pd.read_parquet(os.path.join(DEWEY_PATH, f))
    df = df.loc[df['STATE'] == 'CA']
    df['FILE_DATE'] = pd.to_datetime(df['FILE_DATE'], errors='coerce')
    df['PERMIT_DATE'] = pd.to_datetime(df['PERMIT_DATE'], errors='coerce')
    df['FINAL_DATE'] = pd.to_datetime(df['FINAL_DATE'], errors='coerce')
    summ_df = df.groupby(['STATE', 'JURISDICTION', 'CITY']).agg(
        count = ('JURISDICTION', 'count'),  # any rows 
        first_file_date = ('FILE_DATE', 'min'), 
        last_file_date = ('FILE_DATE', 'max'),
        missing_file_date = ('FILE_DATE', lambda x: x.isna().sum()),
        first_permit_date = ('PERMIT_DATE', 'min'),
        last_permit_date = ('PERMIT_DATE', 'max'),
        missing_permit_date = ('PERMIT_DATE', lambda x: x.isna().sum()),
        first_final_date = ('FINAL_DATE', 'min'),
        last_final_date = ('FINAL_DATE', 'max'),
        missing_final_date = ('FINAL_DATE', lambda x: x.isna().sum()),
        missing_status = ('STATUS_NORMALIZED', lambda x: x.isna().sum())
    ).reset_index()
    summ_df['filename'] = f
    summ_dfs.append(summ_df)

summ_df = pd.concat(summ_dfs)
summ_df = summ_df.sort_values(by=['STATE', 'JURISDICTION', 'CITY'], ascending=True).reset_index(drop=True)



Processing building-permits-united-states_3_7_9.snappy.parquet: (734/734) ...  

In [4]:
summ_df.to_csv(os.path.join(DATA_PATH, "dewey_ca_summary_with_filenames.csv"), index=False)
summ_df.to_parquet(os.path.join(DATA_PATH, "dewey_ca_summary_with_filenames.parquet"), index=False)


In [9]:
summ_df2 = summ_df.groupby(['STATE', 'JURISDICTION', 'CITY']).agg(
    count = ('count', 'sum'),
    first_file_date = ('first_file_date', 'min'),
    last_file_date = ('last_file_date', 'max'),
    missing_file_date = ('missing_file_date', 'sum'),
    first_permit_date = ('first_permit_date', 'min'),
    last_permit_date = ('last_permit_date', 'max'),
    missing_permit_date = ('missing_permit_date', 'sum'),
    first_final_date = ('first_final_date', 'min'),
    last_final_date = ('last_final_date', 'max'),
    missing_final_date = ('missing_final_date', 'sum'),
    missing_status = ('missing_status', 'sum')
).reset_index()

summ_df2 = summ_df2.sort_values(by=['STATE', 'JURISDICTION', 'CITY'], ascending=True).reset_index(drop=True)

summ_df2.to_csv(os.path.join(DATA_PATH, "dewey_ca_summary.csv"), index=False, encoding='utf-8')
summ_df2.to_parquet(os.path.join(DATA_PATH, "dewey_ca_summary.parquet"), index=False)




In [8]:
summ_df.loc[summ_df['JURISDICTION'].str.contains('Flintridge')]

,STATE,JURISDICTION,CITY,count,first_file_date,last_file_date,missing_file_date,first_permit_date,last_permit_date,missing_permit_date,first_final_date,last_final_date,missing_final_date,missing_status,filename
10588,CA,La Cañada Flintridge,La Cañada Flintridge,17537,2018-09-24,2025-10-06,0,2018-09-27,2025-10-02,5253,2018-10-02,2025-10-03,7612,1268,building-permits-united-states_2_2_2.snappy.pa...
